<a href="https://colab.research.google.com/github/shribr/Bricky/blob/main/Tools/dinov2-embeddings/dinov2_retrieval_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DINOv2 Minifigure Retrieval Prototype

Evaluate DINOv2 as a drop-in replacement for the shipped SimCLR torso encoder.

**Runtime:** Select **Runtime → Change runtime type → GPU** (free T4 is enough; A100 finishes the full catalog in ~3 min).

## 1. Clone repo & install dependencies

In [ ]:
MODEL = "dinov2_vits14"  # Options: dinov2_vits14, dinov2_vitb14, dinov2_vitl14

import os, pathlib
from google.colab import userdata

# Clone the repo (has code, 14K renders, catalog — everything needed)
REPO = "/content/Bricky"
if not os.path.isdir(REPO):
    # Use a GitHub token so we can push outputs back later
    token = userdata.get("GITHUB_TOKEN")
    !git clone https://{token}@github.com/shribr/Bricky.git {REPO}
else:
    print(f"Repo already cloned at {REPO}")

os.chdir(REPO)

# Sanity checks
assert os.path.isfile(f"{REPO}/Tools/dinov2-embeddings/embed_catalog.py"), (
    f"Repo structure wrong — expected embed_catalog.py in {REPO}/Tools/dinov2-embeddings/"
)

# Helper: absolute path builder
def P(*parts):
    """Join parts under REPO and return an absolute path string."""
    return str(pathlib.Path(REPO, *parts))

TOOLS   = P("Tools", "dinov2-embeddings")
INDEX   = P("Tools", "dinov2-embeddings", "index", MODEL)
REPORTS = P("Tools", "dinov2-embeddings", "reports")

print(f"REPO:    {REPO}")
print(f"CWD:     {os.getcwd()}")
print(f"MODEL:   {MODEL}")
print(f"TOOLS:   {TOOLS}")
print(f"INDEX:   {INDEX}")
print(f"REPORTS: {REPORTS}")


In [ ]:
!pip install -q -r {TOOLS}/requirements.txt

import os
for d in [INDEX, REPORTS, P("Tools", "dinov2-embeddings", "eval_set")]:
    os.makedirs(d, exist_ok=True)
    print(f"  ✓ {d}")
print("Output directories ready.")


## 2. Build the evaluation set

Deterministic given the seed. Creates 200 figures × 8 variants = 1 600 query images.

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, "-u",
    P("Tools", "dinov2-embeddings", "build_eval_set.py"),
    "--count", "200", "--variants-per-figure", "8", "--seed", "42"
], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})


## 3. Embed the catalog with DINOv2

Start with **ViT-S** (fastest, smallest bundle). If recall is too low, re-run with `dinov2_vitb14` or `dinov2_vitl14`.

In [ ]:
# MODEL is set in the setup cell above. Override here if needed:
# MODEL = "dinov2_vits14"
# MODEL = "dinov2_vitb14"
# MODEL = "dinov2_vitl14"

# Recompute paths if MODEL was changed
INDEX = P("Tools", "dinov2-embeddings", "index", MODEL)
os.makedirs(INDEX, exist_ok=True)
print(f"Using model: {MODEL}")
print(f"INDEX:       {INDEX}")


In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, "-u",
    P("Tools", "dinov2-embeddings", "embed_catalog.py"),
    "--model", MODEL,
    "--batch-size", "64",
    "--out", INDEX
], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})


## 4. Evaluate DINOv2 retrieval

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, "-u",
    P("Tools", "dinov2-embeddings", "evaluate_retrieval.py"),
    "--index", INDEX,
    "--report", f"{REPORTS}/{MODEL}.json"
], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})


## 5. Score the shipped SimCLR index (baseline)

Requires `Tools/torso-embeddings/out/torso_encoder.pt` to be present.

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, "-u",
    P("Tools", "dinov2-embeddings", "compare_existing.py"),
    "--report", f"{REPORTS}/simclr_shipped.json"
], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})


## 6. Compare reports

Load both JSON reports and compare `recall@1`, `recall@5`, `recall@10`.

| recall@5 | Interpretation |
|----------|----------------|
| < 0.20   | Encoder is effectively blind — likely a crop/preprocessing bug |
| 0.20–0.50 | Weak (expected for current SimCLR) |
| 0.50–0.75 | Usable as a top-8 candidate list |
| > 0.75   | Ship it |

In [ ]:
import json, pathlib

def load_report(path):
    with open(path) as f:
        return json.load(f)

dino_report = None
dino_path = pathlib.Path(f"{REPORTS}/{MODEL}.json")
simclr_path = pathlib.Path(f"{REPORTS}/simclr_shipped.json")

if not dino_path.exists():
    print(f"Report not found: {dino_path}")
    print(f"Run the evaluation cells above first to generate the {MODEL} report.")
else:
    dino_report = load_report(dino_path)
    print(f"=== DINOv2 ({MODEL}) ===")
    for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
        print(f"  {k}: {dino_report.get(k, 'N/A')}")

    if simclr_path.exists():
        simclr_report = load_report(simclr_path)
        print(f"\n=== SimCLR (shipped) ===")
        for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
            print(f"  {k}: {simclr_report.get(k, 'N/A')}")
    else:
        print("\nSimCLR report not found — skipping comparison.")


## 7. Inspect failures

The `sample_failures` field lists the 20 worst cases. If the top-5 for a failed query is full of figures with the same torso color but different prints, the encoder is confusing "color" for "identity" — try a stronger backbone or fine-tuning. If the top-5 looks random, there's a preprocessing mismatch.

In [ ]:
if dino_report is None:
    print("No DINOv2 report loaded — run the evaluation cells first.")
else:
    failures = dino_report.get("sample_failures", [])
    print(f"{len(failures)} sample failures:\n")
    for i, f in enumerate(failures[:10], 1):
        print(f"{i}. {f}")


## 8. Real-Photo Evaluation

Score the index against 27 real iPhone photos of LEGO minifigures (wood-grain background, shadows, etc.) instead of synthetic variants. This tests the domain gap between catalog renders and real-world captures.

In [ ]:
import subprocess, sys, os
env = {**os.environ, "PYTHONUNBUFFERED": "1"}

# Build real-photo eval set
subprocess.run([sys.executable, "-u",
    P("Tools", "dinov2-embeddings", "ingest_real_photos.py"), "eval",
    "--mapping", P("Tools", "dinov2-embeddings", "real_photos", "mapping.json")
], check=True, env=env)

# Evaluate against real photos
subprocess.run([sys.executable, "-u",
    P("Tools", "dinov2-embeddings", "evaluate_retrieval.py"),
    "--index", INDEX,
    "--eval", P("Tools", "dinov2-embeddings", "real_photos", "eval"),
    "--report", f"{REPORTS}/{MODEL}_real_photos.json"
], check=True, env=env)


In [ ]:
real_report = None
real_path = pathlib.Path(f"{REPORTS}/{MODEL}_real_photos.json")

if not real_path.exists():
    print(f"Real-photo report not found: {real_path}")
    print("Run the real-photo eval cell above first.")
elif dino_report is None:
    print("No synthetic DINOv2 report loaded — run evaluation cells first.")
else:
    real_report = load_report(real_path)
    print(f"=== Real Photos vs {MODEL} (synthetic eval) ===")
    print(f"{'Metric':<12} {'Synthetic':>10} {'Real Photo':>10}")
    print("-" * 34)
    for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
        synth = dino_report.get(k, "N/A")
        real = real_report.get(k, "N/A")
        synth_s = f"{synth:.3f}" if isinstance(synth, float) else str(synth)
        real_s = f"{real:.3f}" if isinstance(real, float) else str(real)
        print(f"{k:<12} {synth_s:>10} {real_s:>10}")


## 9. Augment Index with Real Photos

Add DINOv2 embeddings of the real photos to the index so the same figure has both a catalog render vector AND a real-world photo vector. This should improve recall when users scan real figures.

In [ ]:
import subprocess, sys, os
INDEX_AUG = P("Tools", "dinov2-embeddings", "index", f"{MODEL}_augmented")
subprocess.run([sys.executable, "-u",
    P("Tools", "dinov2-embeddings", "ingest_real_photos.py"), "augment",
    "--mapping", P("Tools", "dinov2-embeddings", "real_photos", "mapping.json"),
    "--index", INDEX,
    "--out", INDEX_AUG
], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})


In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, "-u",
    P("Tools", "dinov2-embeddings", "evaluate_retrieval.py"),
    "--index", INDEX_AUG,
    "--eval", P("Tools", "dinov2-embeddings", "real_photos", "eval"),
    "--report", f"{REPORTS}/{MODEL}_augmented_real.json"
], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})

aug_path = pathlib.Path(f"{REPORTS}/{MODEL}_augmented_real.json")
if not aug_path.exists():
    print(f"Augmented report not found: {aug_path}")
elif real_report is None:
    print("No real-photo baseline loaded — run previous cells first.")
else:
    aug_report = load_report(aug_path)
    print(f"\n=== Impact of Augmentation (real-photo queries) ===")
    print(f"{'Metric':<12} {'Before':>10} {'After':>10} {'Delta':>10}")
    print("-" * 45)
    for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
        before = real_report.get(k, 0)
        after = aug_report.get(k, 0)
        if isinstance(before, (int, float)) and isinstance(after, (int, float)):
            delta = after - before
            print(f"{k:<12} {before:>10.3f} {after:>10.3f} {delta:>+10.3f}")
        else:
            print(f"{k:<12} {before!s:>10} {after!s:>10}")


## 10. Cross-Source Eval: BrickLink vs Rebrickable

The index is built from **Rebrickable** renders. This section evaluates retrieval using **BrickLink** renders as queries — a completely different rendering pipeline (different camera angle, lighting, head color, aspect ratio). This tests whether the encoder generalises across render styles, which is a much harder and more realistic test than same-source synthetic augmentation.

**Prerequisites:**
- A Rebrickable API key (free at https://rebrickable.com/api/)
- Run `fetch_bricklink_images.py fetch` to download BrickLink renders
- Run `fetch_bricklink_images.py eval` to build the eval set

In [ ]:
# Install cloudscraper (needed for Rebrickable page scraping)
!pip install -q cloudscraper

import subprocess, sys, os
env = {**os.environ, "PYTHONUNBUFFERED": "1"}
BL_IMAGES = P("Tools", "dinov2-embeddings", "bricklink_images")
BL_EVAL = P("Tools", "dinov2-embeddings", "eval_bricklink")

# Step 1: Fetch BrickLink renders
subprocess.run([sys.executable, "-u",
    P("Tools", "dinov2-embeddings", "fetch_bricklink_images.py"), "fetch",
    "--figures", "200", "--seed", "42",
    "--out", BL_IMAGES
], check=True, env=env)

# Step 2: Build cross-source eval set
subprocess.run([sys.executable, "-u",
    P("Tools", "dinov2-embeddings", "fetch_bricklink_images.py"), "eval",
    "--images", BL_IMAGES,
    "--out", BL_EVAL
], check=True, env=env)


In [ ]:
# Step 3: Run cross-source retrieval eval
import subprocess, sys, os
subprocess.run([sys.executable, "-u",
    P("Tools", "dinov2-embeddings", "evaluate_retrieval.py"),
    "--index", INDEX,
    "--eval", BL_EVAL,
    "--report", f"{REPORTS}/{MODEL}_cross_source.json"
], check=True, env={**os.environ, "PYTHONUNBUFFERED": "1"})

cross_path = pathlib.Path(f"{REPORTS}/{MODEL}_cross_source.json")
if not cross_path.exists():
    print(f"Cross-source report not found: {cross_path}")
else:
    cross_report = load_report(cross_path)
    synth_val = dino_report if dino_report else {}
    print(f"\n=== Cross-Source Eval: BrickLink queries -> Rebrickable index ===")
    print(f"{'Metric':<12} {'Synthetic':>10} {'CrossSrc':>10} {'Delta':>10}")
    print("-" * 45)
    for k in ["recall@1", "recall@5", "recall@10", "recall@50"]:
        synth = synth_val.get(k, 0)
        cross = cross_report.get(k, 0)
        if isinstance(synth, (int, float)) and isinstance(cross, (int, float)):
            delta = cross - synth
            print(f"{k:<12} {synth:>10.3f} {cross:>10.3f} {delta:>+10.3f}")
        else:
            print(f"{k:<12} {synth!s:>10} {cross!s:>10}")


In [ ]:
# Step 4: Visual comparison — same figure, two render sources
from PIL import Image
from IPython.display import display
import json, random, pathlib

bl_index_path = pathlib.Path(f"{BL_IMAGES}/index.json")
if not bl_index_path.exists():
    print(f"BrickLink index not found: {bl_index_path}")
    print("Run the BrickLink fetch cell (Step 1) first.")
else:
    bl_index = json.loads(bl_index_path.read_text())
    sample_ids = random.Random(42).sample(list(bl_index.keys()), min(4, len(bl_index)))

    for fid in sample_ids:
        rb_path = P("Bricky", "Resources", "MinifigImages", f"{fid}.jpg")
        bl_path = f"{BL_IMAGES}/{fid}.png"
        try:
            rb = Image.open(rb_path).convert("RGB")
            bl = Image.open(bl_path).convert("RGB")
            h = max(rb.height, bl.height)
            canvas = Image.new("RGB", (rb.width + bl.width + 20, h + 30), (255, 255, 255))
            canvas.paste(rb, (0, 30 + (h - rb.height) // 2))
            canvas.paste(bl, (rb.width + 20, 30 + (h - bl.height) // 2))
            print(f"\n{fid} — Left: Rebrickable | Right: BrickLink")
            display(canvas)
        except FileNotFoundError:
            print(f"  {fid}: image not found, skipping")


## 11. Save outputs to GitHub

Commit all generated outputs (embeddings, reports, eval sets) back to the repo.
Requires a `GITHUB_TOKEN` secret in Colab (Settings → Secrets).


In [ ]:
import subprocess, os

os.chdir(REPO)

# Configure git identity for the commit
!git config user.email "colab@users.noreply.github.com"
!git config user.name "Colab Pipeline"

# Stage all generated outputs
output_dirs = [
    "Tools/dinov2-embeddings/index/",
    "Tools/dinov2-embeddings/reports/",
    "Tools/dinov2-embeddings/eval/",
    "Tools/dinov2-embeddings/eval_set/",
    "Tools/dinov2-embeddings/eval_bricklink/",
    "Tools/dinov2-embeddings/bricklink_images/",
    "Tools/dinov2-embeddings/real_photos/",
]
for d in output_dirs:
    full = os.path.join(REPO, d)
    if os.path.isdir(full):
        !git add -A {d}
        print(f"  staged: {d}")
    else:
        print(f"  skip (not found): {d}")

# Show what will be committed
!git status --short

# Commit and push
!git diff --cached --stat
!git commit -m "chore: DINOv2 {MODEL} eval outputs from Colab" || echo "Nothing to commit."
!git push origin main
print("\nAll outputs pushed to GitHub.")


<a href="https://colab.research.google.com/github/shribr/Bricky/blob/main/Tools/dinov2-embeddings/dinov2_retrieval_prototype.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>